# Laboratorio 4 – Segmentación de imágenes con técnicas de agrupación

**Grupo:** G11

**Integrantes:**
- Juan Diego Ramirez – 202410855
- Sebastian Vargas Otero – 202323722

---
**Caso DermAlpes.** Se busca delimitar automáticamente la lesión en imágenes dermatoscópicas **sin entrenar un modelo supervisado**: cada píxel se representa por su color y su posición, se agrupa con K-medias y, a partir de los grupos, se construye una máscara binaria de la lesión. Las máscaras de referencia solo se usan para **evaluar**.

**Fuente de los datos.** ISIC 2016 – *Skin Lesion Analysis Toward Melanoma Detection*, tarea 1 (segmentación de lesiones). https://challenge.isic-archive.com/landing/2016/37/
Gutman, D., Codella, N. C. F., Celebi, E., Helba, B., Marchetti, M., Mishra, N., & Halpern, A. (2016). *Skin Lesion Analysis toward Melanoma Detection: A Challenge at the International Symposium on Biomedical Imaging (ISBI) 2016*. arXiv:1605.01397.

**Protocolo.** 15 imágenes de **ajuste** (para elegir β, λ y el criterio de máscara) y 5 de **prueba**, que solo se consultan al final (sección 5).

## 0. Configuración

Estructura esperada de los datos (puede cambiarse con la variable de entorno `LAB4_DATA`):

```
data/
├── imagenes/  ISIC_0000085.jpg, ...
└── mascaras/  ISIC_0000085_Segmentation.png, ...
```

In [ ]:
import os
import glob
import time
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy import ndimage as ndi

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, f1_score, v_measure_score,
                             precision_score, recall_score)

DATA_DIR = os.environ.get("LAB4_DATA", "data")
IMG_DIR = os.path.join(DATA_DIR, "imagenes")
MASK_DIR = os.path.join(DATA_DIR, "mascaras")

SEED = 42             # semilla para K-medias y para la muestra de la silueta
MAX_LADO = 256        # las imágenes se reducen a 256 px en su lado mayor
N_MUESTRA_SIL = 3000  # píxeles muestreados para calcular la silueta
K_RANGO = range(2, 9)
N_INIT = 4

IDS_AJUSTE = ["ISIC_0000085", "ISIC_0000086", "ISIC_0000089", "ISIC_0000094", "ISIC_0000108",
              "ISIC_0000114", "ISIC_0000116", "ISIC_0000122", "ISIC_0000133", "ISIC_0000137",
              "ISIC_0000150", "ISIC_0000153", "ISIC_0000156", "ISIC_0000157", "ISIC_0000112"]
IDS_PRUEBA = ["ISIC_0000087", "ISIC_0000105", "ISIC_0000110", "ISIC_0000123", "ISIC_0000128"]

sns.set_theme(style="whitegrid")

## 1. Pipeline de preparación y representación

### 1.1 Carga de imágenes y máscaras
Las imágenes se reducen (interpolación bilineal) para que K-medias sea manejable: con 256 px en el lado mayor quedan ≈ 49 000 píxeles por imagen. La máscara se redimensiona al mismo tamaño con **vecino más cercano** para conservar sus valores binarios, y se convierte de {0, 255} a {0, 1}.

In [ ]:
def buscar_archivo(carpeta, isic_id, sufijos):
    for sufijo in sufijos:
        hits = glob.glob(os.path.join(carpeta, isic_id + sufijo))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"No se encontró {isic_id} en {carpeta}")


def cargar_par(isic_id, max_lado=MAX_LADO):
    img = Image.open(buscar_archivo(IMG_DIR, isic_id, [".jpg", ".jpeg", ".png"])).convert("RGB")
    mask = Image.open(buscar_archivo(MASK_DIR, isic_id,
                                     ["_Segmentation.png", "_segmentation.png", ".png"])).convert("L")
    tam_original = img.size
    escala = max_lado / max(img.size)
    if escala < 1:
        img = img.resize((round(img.width * escala), round(img.height * escala)), Image.BILINEAR)
    # vecino más cercano para que la máscara siga siendo binaria
    mask = mask.resize(img.size, Image.NEAREST)
    mask = (np.asarray(mask) > 127).astype(np.uint8)  # 255 -> 1, 0 -> 0
    return np.asarray(img), mask, tam_original


datos = {}
for isic_id in IDS_AJUSTE + IDS_PRUEBA:
    img, mask, tam = cargar_par(isic_id)
    datos[isic_id] = {"img": img, "mask": mask}

resumen = pd.DataFrame([
    {"id": i, "conjunto": "ajuste" if i in IDS_AJUSTE else "prueba",
     "alto": d["img"].shape[0], "ancho": d["img"].shape[1],
     "valores_mascara": tuple(np.unique(d["mask"])), "% lesión": 100 * d["mask"].mean()}
    for i, d in datos.items()
])
resumen.round(1)

In [ ]:
# Vista general de las imágenes de ajuste con el contorno de la máscara de referencia
fig, axes = plt.subplots(3, 5, figsize=(16, 9))
for ax, isic_id in zip(axes.ravel(), IDS_AJUSTE):
    ax.imshow(datos[isic_id]["img"])
    ax.contour(datos[isic_id]["mask"], levels=[0.5], colors="lime", linewidths=1.2)
    ax.set_title(isic_id, fontsize=9)
    ax.axis("off")
fig.suptitle("Imágenes de ajuste (contorno verde = máscara de referencia)")
plt.tight_layout()
plt.show()

### 1.2 Representación [β·R, β·G, β·B, λ·x, λ·y]
`RepresentacionPixeles` es un transformador de Scikit-learn que recibe una imagen `H×W×3` y devuelve una matriz `(H·W)×5` con un píxel por fila. Color y coordenadas se normalizan a [0, 1] (color / 255; x / (W−1); y / (H−1)) y luego se ponderan por β y λ, que son hiperparámetros del transformador y se pueden cambiar con `set_params` sin modificar la implementación.

In [ ]:
class RepresentacionPixeles(BaseEstimator, TransformerMixin):
    # Imagen H x W x 3 -> matriz (H*W) x 5 con [beta*R, beta*G, beta*B, lam*x, lam*y]

    def __init__(self, beta=1.0, lam=0.0):
        self.beta = beta
        self.lam = lam

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        img = np.asarray(X, dtype=float)
        h, w, _ = img.shape
        color = img.reshape(-1, 3) / 255.0
        yy, xx = np.mgrid[0:h, 0:w]
        x = xx.ravel() / (w - 1)
        y = yy.ravel() / (h - 1)
        return np.column_stack([self.beta * color, self.lam * x, self.lam * y])


def construir_pipeline(beta, lam, k):
    return Pipeline([
        ("representacion", RepresentacionPixeles(beta=beta, lam=lam)),
        ("kmeans", KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED)),
    ])


# Verificación rápida
X_demo = RepresentacionPixeles(beta=1, lam=0.5).fit_transform(datos[IDS_AJUSTE[0]]["img"])
print("Forma:", X_demo.shape)
print("Mínimos:", X_demo.min(axis=0).round(3), " Máximos:", X_demo.max(axis=0).round(3))

## 2. Agrupación y selección de K

### 2.1 Combinaciones de β y λ
Como K-medias y la silueta solo dependen de distancias relativas, lo que importa es la razón **λ/β**. Se fija β = 1 y se varía λ, más la combinación de solo posición (β = 0, λ = 1). Con color y coordenadas en [0, 1], una razón λ/β = 1 da aproximadamente el mismo peso a una diferencia de color "completa" que a cruzar toda la imagen.

In [ ]:
COMBINACIONES = {
    "solo_color (λ/β=0)":   (1.0, 0.0),
    "λ/β=0.1":              (1.0, 0.1),
    "λ/β=0.25":             (1.0, 0.25),
    "λ/β=0.5":              (1.0, 0.5),
    "λ/β=0.75":             (1.0, 0.75),
    "λ/β=1":                (1.0, 1.0),
    "λ/β=1.5":              (1.0, 1.5),
    "λ/β=2":                (1.0, 2.0),
    "λ/β=4":                (1.0, 4.0),
    "solo_posición (β=0)":  (0.0, 1.0),
}
pd.DataFrame(COMBINACIONES, index=["β", "λ"]).T

### 2.2 Selección de K con el coeficiente de silueta
Para cada imagen y combinación se ajusta K-medias con K = 2…8 sobre **todos** los píxeles, y la silueta se calcula sobre una **muestra reproducible de 3 000 píxeles** (semilla 42; la misma muestra para todos los K y combinaciones de una imagen). Se elige el K con mayor silueta. Calcular la silueta sobre los ≈ 49 000 píxeles sería O(n²) en memoria y tiempo.

Se guardan además las etiquetas de todos los K para poder analizar después la relación K – silueta – Dice (pregunta 2), pero **la decisión de K usa únicamente la silueta**.

In [ ]:
def indices_muestra(n_pixeles, n=N_MUESTRA_SIL, seed=SEED):
    rng = np.random.default_rng(seed)
    return rng.choice(n_pixeles, size=min(n, n_pixeles), replace=False)


def agrupar_todos_k(img, beta, lam, k_rango=K_RANGO):
    # Ajusta el pipeline para cada K y devuelve {k: (etiquetas, silueta)}
    X = RepresentacionPixeles(beta, lam).fit_transform(img)
    idx = indices_muestra(X.shape[0])
    salida = {}
    for k in k_rango:
        pipe = construir_pipeline(beta, lam, k).fit(img)
        etiquetas = pipe.named_steps["kmeans"].labels_
        sil = silhouette_score(X[idx], etiquetas[idx])
        salida[k] = (etiquetas, sil)
    return salida


def k_por_silueta(por_k):
    return max(por_k, key=lambda k: por_k[k][1])

In [ ]:
t0 = time.time()
agrupaciones = {}  # (id, combinación) -> {k: (etiquetas, silueta)}
for isic_id in IDS_AJUSTE:
    for nombre, (beta, lam) in COMBINACIONES.items():
        agrupaciones[(isic_id, nombre)] = agrupar_todos_k(datos[isic_id]["img"], beta, lam)
    print(f"{isic_id} listo ({time.time() - t0:.0f} s)")

In [ ]:
filas_sil = [{"id": i, "combinación": c, "K": k, "silueta": s}
             for (i, c), por_k in agrupaciones.items() for k, (_, s) in por_k.items()]
df_sil = pd.DataFrame(filas_sil)

k_elegidos = (df_sil.loc[df_sil.groupby(["id", "combinación"])["silueta"].idxmax()]
              .pivot(index="id", columns="combinación", values="K")[list(COMBINACIONES)])
print("K seleccionado por silueta (imágenes de ajuste):")
k_elegidos

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=df_sil, x="K", y="silueta", hue="combinación", marker="o", errorbar=None, ax=ax)
ax.set_title("Silueta promedio (15 imágenes de ajuste) en función de K")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 3. Construcción de la máscara de la lesión

K-medias produce grupos sin significado. Para decidir qué grupos son lesión se calculan estadísticas de cada grupo usando el **color original** de sus píxeles (no el ponderado), de modo que el criterio funciona igual para cualquier β y λ:

- **Luminancia media** `L = 0.299R + 0.587G + 0.114B`: en dermatoscopia la lesión suele ser más oscura que la piel circundante.
- **Fracción en el borde**: proporción de los píxeles del grupo que caen en una franja del 5 % del borde de la imagen. Las esquinas oscuras (viñeteado del dermatoscopio) y los marcos negros son oscuros pero están en el borde; la lesión suele estar centrada.

**Criterio A – Umbral de luminancia.** Se marcan como lesión los grupos cuya luminancia media está por debajo del punto medio entre el grupo más oscuro y el más claro. Con K = 2 equivale a escoger el grupo más oscuro.

**Criterio B – Oscuro, fuera del borde y compacto.** Igual que A, pero se descartan los grupos con más del 40 % de sus píxeles en el borde (si eso descarta todos, se toma el grupo con menor `L + fracción_borde`). Luego se conserva la **componente conexa más grande** y se **rellenan huecos**, porque la lesión es una región única y continua.

In [ ]:
LUM = np.array([0.299, 0.587, 0.114])
MARGEN_BORDE = 0.05
UMBRAL_BORDE = 0.40


def banda_borde(h, w, margen=MARGEN_BORDE):
    m = max(1, int(round(margen * min(h, w))))
    banda = np.zeros((h, w), dtype=bool)
    banda[:m, :] = banda[-m:, :] = True
    banda[:, :m] = banda[:, -m:] = True
    return banda.ravel()


def estadisticas_grupos(img, etiquetas):
    h, w, _ = img.shape
    lum = (img.reshape(-1, 3) / 255.0) @ LUM
    borde = banda_borde(h, w)
    filas = []
    for g in np.unique(etiquetas):
        sel = etiquetas == g
        filas.append({"grupo": g, "fracción": sel.mean(), "luminancia": lum[sel].mean(),
                      "fracción_borde": borde[sel].mean()})
    return pd.DataFrame(filas).set_index("grupo")


def grupos_oscuros(stats):
    umbral = (stats["luminancia"].min() + stats["luminancia"].max()) / 2
    return stats.index[stats["luminancia"] < umbral].tolist() or [stats["luminancia"].idxmin()]


def mayor_componente(mask):
    comp, n = ndi.label(mask)
    if n <= 1:
        return mask
    tamanos = ndi.sum(mask, comp, index=range(1, n + 1))
    return comp == (np.argmax(tamanos) + 1)


def mascara_criterio_A(img, etiquetas):
    stats = estadisticas_grupos(img, etiquetas)
    lesion = grupos_oscuros(stats)
    return np.isin(etiquetas, lesion).reshape(img.shape[:2]).astype(np.uint8)


def mascara_criterio_B(img, etiquetas):
    stats = estadisticas_grupos(img, etiquetas)
    oscuros = grupos_oscuros(stats)
    lesion = [g for g in oscuros if stats.loc[g, "fracción_borde"] <= UMBRAL_BORDE]
    if not lesion:
        lesion = [(stats["luminancia"] + stats["fracción_borde"]).idxmin()]
    mask = np.isin(etiquetas, lesion).reshape(img.shape[:2])
    mask = ndi.binary_fill_holes(mayor_componente(mask))
    return mask.astype(np.uint8)


CRITERIOS = {"A_luminancia": mascara_criterio_A, "B_oscuro_central": mascara_criterio_B}

In [ ]:
# Ejemplo del criterio sobre una imagen de ajuste
ej_id, ej_comb = IDS_AJUSTE[0], "λ/β=0.5"
por_k = agrupaciones[(ej_id, ej_comb)]
k_ej = k_por_silueta(por_k)
et_ej = por_k[k_ej][0]
display(estadisticas_grupos(datos[ej_id]["img"], et_ej).round(3))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(datos[ej_id]["img"]); axes[0].set_title(ej_id)
axes[1].imshow(et_ej.reshape(datos[ej_id]["img"].shape[:2]), cmap="tab10"); axes[1].set_title(f"Grupos ({ej_comb}, K={k_ej})")
axes[2].imshow(mascara_criterio_A(datos[ej_id]["img"], et_ej), cmap="gray"); axes[2].set_title("Criterio A")
axes[3].imshow(mascara_criterio_B(datos[ej_id]["img"], et_ej), cmap="gray"); axes[3].set_title("Criterio B")
for ax in axes:
    ax.contour(datos[ej_id]["mask"], levels=[0.5], colors="lime", linewidths=1)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Ajuste del método (solo imágenes de ajuste)

Para cada imagen, combinación y criterio se calcula:
- **Dice** entre la máscara estimada y la de referencia (`f1_score` sobre las máscaras aplanadas) → criterio principal.
- **V-measure** entre los grupos de K-medias y la máscara de referencia → información complementaria (no depende del criterio de máscara).

También se registra el Dice para cada K (no solo el elegido por silueta) para el análisis; **la selección de la configuración usa solo el K elegido por silueta**.

In [ ]:
def evaluar(mask_est, mask_ref, etiquetas):
    ref, est = mask_ref.ravel(), mask_est.ravel()
    return {"dice": f1_score(ref, est, zero_division=0),
            "v_measure": v_measure_score(ref, etiquetas),
            "precisión": precision_score(ref, est, zero_division=0),
            "recall": recall_score(ref, est, zero_division=0)}


filas, filas_k = [], []
for (isic_id, comb), por_k in agrupaciones.items():
    img, ref = datos[isic_id]["img"], datos[isic_id]["mask"]
    k_sel = k_por_silueta(por_k)
    for k, (etiquetas, sil) in por_k.items():
        for crit, fn in CRITERIOS.items():
            m = evaluar(fn(img, etiquetas), ref, etiquetas)
            fila = {"id": isic_id, "combinación": comb, "criterio": crit, "K": k, "silueta": sil, **m}
            filas_k.append(fila)
            if k == k_sel:
                filas.append(fila)

df_ajuste = pd.DataFrame(filas)      # K elegido por silueta
df_todos_k = pd.DataFrame(filas_k)   # todos los K (solo para análisis)

tabla_ajuste = (df_ajuste.groupby(["combinación", "criterio"])
                .agg(dice_prom=("dice", "mean"), dice_std=("dice", "std"),
                     dice_min=("dice", "min"), v_measure_prom=("v_measure", "mean"),
                     K_prom=("K", "mean"), silueta_prom=("silueta", "mean"))
                .sort_values("dice_prom", ascending=False))
tabla_ajuste.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
for ax, crit in zip(axes, CRITERIOS):
    piv = (df_ajuste[df_ajuste["criterio"] == crit]
           .pivot(index="combinación", columns="id", values="dice").loc[list(COMBINACIONES)])
    sns.heatmap(piv, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax, cbar=False,
                annot_kws={"size": 7})
    ax.set_title(f"Dice por imagen – {crit}")
    ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

### 4.1 Relación K – silueta – Dice

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
prom_k = df_todos_k.groupby(["K", "criterio"])[["silueta", "dice"]].mean().reset_index()
sns.lineplot(data=prom_k, x="K", y="dice", hue="criterio", marker="o", ax=axes[0])
axes[0].plot(prom_k.groupby("K")["silueta"].first(), "k--o", label="silueta")
axes[0].legend(); axes[0].set_title("Promedio sobre imágenes y combinaciones")

sns.scatterplot(data=df_ajuste, x="silueta", y="dice", hue="criterio", alpha=0.6, ax=axes[1])
axes[1].set_title("Silueta vs Dice (K elegido por silueta)")
plt.tight_layout()
plt.show()

# ¿Cuánto Dice se pierde por elegir K con silueta en lugar del K "óptimo" (oráculo, solo para análisis)?
oraculo = df_todos_k.groupby(["id", "combinación", "criterio"])["dice"].max().rename("dice_mejor_K")
comp = df_ajuste.set_index(["id", "combinación", "criterio"])["dice"].to_frame().join(oraculo)
print("Dice promedio con K por silueta vs mejor K posible:")
comp.groupby("criterio").mean().round(3)

### 4.2 Visualización de las agrupaciones
Imagen original y grupos (cada grupo pintado con su color medio) para las dos combinaciones con mayor Dice promedio, solo color y solo posición. El contorno verde es la máscara de referencia.

In [ ]:
ranking_comb = tabla_ajuste.reset_index().drop_duplicates("combinación")["combinación"].tolist()
comb_mostrar = list(dict.fromkeys(ranking_comb[:2] + ["solo_color (λ/β=0)", "solo_posición (β=0)"]))


def imagen_por_grupos(img, etiquetas):
    planos = img.reshape(-1, 3).astype(float)
    salida = np.zeros_like(planos)
    for g in np.unique(etiquetas):
        salida[etiquetas == g] = planos[etiquetas == g].mean(axis=0)
    return salida.reshape(img.shape).astype(np.uint8)


def mostrar_agrupaciones(ids):
    fig, axes = plt.subplots(len(ids), len(comb_mostrar) + 1, figsize=(3.2 * (len(comb_mostrar) + 1), 2.6 * len(ids)))
    for fila, isic_id in zip(axes, ids):
        img, ref = datos[isic_id]["img"], datos[isic_id]["mask"]
        fila[0].imshow(img); fila[0].set_title(isic_id, fontsize=9)
        for ax, comb in zip(fila[1:], comb_mostrar):
            por_k = agrupaciones[(isic_id, comb)]
            k = k_por_silueta(por_k)
            ax.imshow(imagen_por_grupos(img, por_k[k][0]))
            ax.set_title(f"{comb}\nK={k}", fontsize=8)
        for ax in fila:
            ax.contour(ref, levels=[0.5], colors="lime", linewidths=1)
            ax.axis("off")
    plt.tight_layout()
    plt.show()


mostrar_agrupaciones(IDS_AJUSTE[:8])

In [ ]:
mostrar_agrupaciones(IDS_AJUSTE[8:])

### 4.3 Casos de fallo de la mejor configuración

In [ ]:
mejor_comb, mejor_crit = tabla_ajuste.index[0]
print(f"Mejor configuración en ajuste: {mejor_comb} | β, λ = {COMBINACIONES[mejor_comb]} | criterio {mejor_crit}")

det = (df_ajuste[(df_ajuste["combinación"] == mejor_comb) & (df_ajuste["criterio"] == mejor_crit)]
       .sort_values("dice").reset_index(drop=True))
display(det.round(3))


def mostrar_mascaras(ids, comb, crit, agrup):
    fig, axes = plt.subplots(len(ids), 4, figsize=(13, 3 * len(ids)), squeeze=False)
    for fila, isic_id in zip(axes, ids):
        img, ref = datos[isic_id]["img"], datos[isic_id]["mask"]
        por_k = agrup[(isic_id, comb)]
        k = k_por_silueta(por_k)
        et = por_k[k][0]
        est = CRITERIOS[crit](img, et)
        dice = f1_score(ref.ravel(), est.ravel(), zero_division=0)
        fila[0].imshow(img); fila[0].set_title(isic_id, fontsize=9)
        fila[1].imshow(imagen_por_grupos(img, et)); fila[1].set_title(f"Grupos K={k}", fontsize=9)
        fila[2].imshow(est, cmap="gray"); fila[2].set_title(f"Estimada (Dice={dice:.2f})", fontsize=9)
        fila[3].imshow(ref, cmap="gray"); fila[3].set_title("Referencia", fontsize=9)
        for ax in fila:
            ax.axis("off")
    plt.tight_layout()
    plt.show()


print("Tres peores imágenes de ajuste:")
mostrar_mascaras(det["id"].head(3).tolist(), mejor_comb, mejor_crit, agrupaciones)

### 4.4 Configuración final

> **TODO (grupo):** justificar la elección con base en la tabla anterior: Dice promedio, dispersión/Dice mínimo, V-measure y lo observado en los casos de fallo. Si dos configuraciones están empatadas, preferir la más estable (menor desviación, mayor Dice mínimo). Si después de la verificación con IAG (sección 6) se decide cambiar la configuración, hacerlo **antes** de ejecutar la sección 5.

In [ ]:
CONFIG_FINAL = {
    "combinación": mejor_comb,
    "beta": COMBINACIONES[mejor_comb][0],
    "lam": COMBINACIONES[mejor_comb][1],
    "criterio": mejor_crit,
    "K": "elegido por silueta en [2, 8]",
}
CONFIG_FINAL

## 5. Evaluación en las imágenes de prueba

Se aplica `CONFIG_FINAL` **sin modificaciones**. Es la primera vez que se usan las máscaras de prueba.

In [ ]:
beta_f, lam_f, crit_f = CONFIG_FINAL["beta"], CONFIG_FINAL["lam"], CONFIG_FINAL["criterio"]
agrup_prueba, filas_prueba = {}, []
for isic_id in IDS_PRUEBA:
    img, ref = datos[isic_id]["img"], datos[isic_id]["mask"]
    por_k = agrupar_todos_k(img, beta_f, lam_f)
    agrup_prueba[(isic_id, CONFIG_FINAL["combinación"])] = por_k
    k = k_por_silueta(por_k)
    etiquetas, sil = por_k[k]
    filas_prueba.append({"id": isic_id, "K": k, "silueta": sil,
                         **evaluar(CRITERIOS[crit_f](img, etiquetas), ref, etiquetas)})

df_prueba = pd.DataFrame(filas_prueba).set_index("id")
df_prueba.loc["Promedio"] = df_prueba.mean()
df_prueba.round(3)

In [ ]:
mostrar_mascaras(IDS_PRUEBA, CONFIG_FINAL["combinación"], crit_f, agrup_prueba)

In [ ]:
dice_ajuste = det["dice"].mean()
dice_prueba = df_prueba.loc[IDS_PRUEBA, "dice"].mean()
print(f"Dice promedio ajuste: {dice_ajuste:.3f}  (n=15)")
print(f"Dice promedio prueba: {dice_prueba:.3f}  (n=5)")

> **TODO (grupo):** comentar aciertos y errores principales en prueba (qué imágenes funcionan, cuáles no y por qué: vello, viñeteado, bajo contraste, lesión multicolor, burbujas, reglas/marcas, etc.).

## 6. Uso de herramientas de IA generativa

### 6.1 Actividad de verificación (obligatoria)
Prompt sugerido (no adjuntar imágenes ni mencionar el conjunto de datos):

> *Tengo fotografías de lesiones de piel tomadas con un dermatoscopio: la lesión suele ser una mancha marrón u oscura, más o menos centrada, sobre piel rosada o beige; algunas imágenes tienen esquinas oscuras circulares por el lente, vello, burbujas o reflejos. Quiero segmentar la lesión sin supervisión con K-medias, representando cada píxel como [β·R, β·G, β·B, λ·x, λ·y] con color y coordenadas normalizados a [0, 1]. ¿Qué valores de β, λ y K recomiendas, y qué criterio automático usarías para decidir qué grupos son la lesión y construir una máscara binaria?*

Adjuntar el historial completo de la conversación. Luego, poner la recomendación en la celda siguiente y evaluarla en las imágenes de ajuste.

In [ ]:
# TODO: reemplazar con lo que recomiende la IAG
RECOMENDACION_IAG = {"beta": 1.0, "lam": 0.3, "K": 2, "criterio": "A_luminancia"}  # K=None -> silueta

filas_iag = []
for isic_id in IDS_AJUSTE:
    img, ref = datos[isic_id]["img"], datos[isic_id]["mask"]
    if RECOMENDACION_IAG["K"] is None:
        por_k = agrupar_todos_k(img, RECOMENDACION_IAG["beta"], RECOMENDACION_IAG["lam"])
        k = k_por_silueta(por_k)
        etiquetas = por_k[k][0]
    else:
        k = RECOMENDACION_IAG["K"]
        pipe = construir_pipeline(RECOMENDACION_IAG["beta"], RECOMENDACION_IAG["lam"], k).fit(img)
        etiquetas = pipe.named_steps["kmeans"].labels_
    est = CRITERIOS[RECOMENDACION_IAG["criterio"]](img, etiquetas)
    filas_iag.append({"id": isic_id, "K": k, **evaluar(est, ref, etiquetas)})

df_iag = pd.DataFrame(filas_iag).set_index("id")
print(f"Dice promedio recomendación IAG: {df_iag['dice'].mean():.3f}  vs  configuración propia: {dice_ajuste:.3f}")
df_iag.round(3)

**¿La recomendación funcionó en nuestras imágenes? ¿En qué acertó y en qué falló? ¿Por qué la IAG no podía anticipar el comportamiento del método?**

> TODO (grupo)

### 6.2 Declaración de uso
- **Herramienta(s) y tipo de uso:** TODO (p. ej. Claude Code – generación inicial del esqueleto del notebook; ChatGPT/Claude – actividad de verificación).
- **Prompts principales:** TODO
- **Análisis crítico del resultado:** TODO (responder al menos dos de las preguntas del enunciado).
- **Aportes propios:** TODO (qué se desarrolló, modificó o decidió por el grupo).

## 7. Análisis de resultados

> Responder con valores concretos de las tablas y figuras anteriores.

**1. ¿Qué efecto tuvo incluir la posición de los píxeles?** Comparar `solo_color` contra las razones λ/β intermedias y `solo_posición` (tabla de la sección 4 y mapa de calor). Apoyarse en las figuras de 4.2: imágenes donde la posición ayuda (p. ej. elimina esquinas oscuras o vello lejos del centro) y donde perjudica (lesiones grandes o descentradas que se parten en bloques espaciales).

> TODO

**2. Relación entre K, silueta y Dice.** Ver 4.1: la silueta mide la separación de los grupos en el espacio de características, no si alguno coincide con la lesión; un K pequeño suele dar alta silueta pero puede mezclar lesión y piel oscura, y un K grande fragmenta la lesión. Usar la comparación "K por silueta vs mejor K".

> TODO

**3. Ajuste vs prueba.** Comparar Dice promedio de ajuste y prueba (sección 5). Explicar por qué modificar el método tras ver las máscaras de prueba convierte la prueba en ajuste y sesga optimistamente la estimación.

> TODO

**4. Dos limitaciones del modelo.** Candidatas: (i) RGB es sensible a iluminación/viñeteado y no separa bien lesiones claras o con bajo contraste; (ii) no hay información de textura ni forma, por lo que vello, reflejos o burbujas se agrupan con la lesión; (iii) el criterio "más oscuro" falla en lesiones multicolor (regiones rosadas/azules); (iv) la posición asume lesión centrada y compacta.

> TODO

**5. Falsos positivos / negativos, sesgos y generalización.** FP → sobrestima tamaño/asimetría de la lesión, puede alarmar; FN → recorta zonas de la lesión (bordes irregulares, regiones claras), subestima tamaño y puede ocultar rasgos relevantes. Sesgos: ISIC 2016 tiene predominantemente fototipos claros; en piel más oscura el contraste lesión–piel cambia y el criterio por luminancia puede fallar. Diferencias de dermatoscopio, iluminación, resolución y protocolo de captura en los centros de salud de DermAlpes.

> TODO

## 8. Bonos
> Pendiente (opcional): Bono A (PCA/t-SNE del espacio de representación) y Bono B (mean shift).